In [15]:
# import importlib.util

# # Install only if missing (Jupyter-friendly)
# if importlib.util.find_spec("langchain_chroma") is None:
#     %pip install -q langchain-chroma

import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Plus

import importlib
import models

importlib.reload(models)

from models import ChunkMetadata, RankingKeywords

print("models file:", models.__file__)
print("RankingKeywords fields:", models.RankingKeywords.model_fields.keys())

models file: /home/siddhant-gond/Desktop/Agentic-RAG-with-LangGraph-and-Ollama-main./rag/models.py
RankingKeywords fields: dict_keys(['keywords'])


In [16]:
# configurations

load_dotenv(dotenv_path=Path("rag/.env"))

CHROMA_DB = "chroma_db"
COLLECTION_NAME ="financial_docs"
EMBEDDING_MODEL = "nomic-embed-text"
BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
LLM_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:latest")


In [17]:
try:
    embedding_model = OllamaEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=BASE_URL,
        num_ctx=8192,
    )
    if embedding_model is None:
        raise ValueError("Embedding model couldn't be loaded")
except Exception as e:
    raise RuntimeError(f"Failed to initialize embedding model: {e}") from e


try:
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=CHROMA_DB,
    )
    if vector_store is None:
        raise ValueError("Vector store couldn't be initialized")
except Exception as e:
    raise RuntimeError(f"Failed to initialize vector store: {e}") from e


In [18]:
import json
from urllib.request import urlopen
from urllib.error import URLError

# initializing the llm (with model/host fallback)

def _available_models(base_url: str) -> set[str]:
    try:
        with urlopen(f"{base_url}/api/tags", timeout=5) as r:
            payload = json.loads(r.read().decode("utf-8"))
        return {m.get("name", "") for m in payload.get("models", [])}
    except Exception:
        return set()

# Fix common typo: llama.3.2:latest -> llama3.2:latest
model_candidates = [
    LLM_MODEL,
    LLM_MODEL.replace("llama.", "llama"),
    "llama3.2:latest",
    "llama3.2",
]
model_candidates = list(dict.fromkeys(model_candidates))  # de-duplicate, keep order

base_url_candidates = [BASE_URL, "http://localhost:11434"]
base_url_candidates = list(dict.fromkeys(base_url_candidates))

LLM = None
response = None
last_error = None

for base in base_url_candidates:
    available = _available_models(base)
    for model_name in model_candidates:
        # If tags endpoint is reachable, skip models that are definitely absent
        if available and model_name not in available:
            continue
        try:
            test_llm = ChatOllama(model=model_name, base_url=base)
            test_response = test_llm.invoke("Hello")
            LLM = test_llm
            response = test_response
            LLM_MODEL = model_name  # keep corrected model in notebook state
            break
        except Exception as e:
            last_error = e
    if LLM is not None:
        break

if LLM is None:
    raise RuntimeError(
        f"Failed to initialize LLM. Last error: {last_error}\n"
        "Install a valid model, e.g.: ollama pull llama3.2:latest"
    )

response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-15T15:49:45.298268318Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6611770720, 'load_duration': 5236674398, 'prompt_eval_count': 26, 'prompt_eval_duration': 885660171, 'eval_count': 10, 'eval_duration': 468871371, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019d91d5-74fb-7510-a75a-123b3a6c1a0f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36})

In [19]:
### EXTRACT FILTERS AND RANKING KEYWORDS

# Fast pre-check patterns: only run LLM extraction if query looks relevant
COMPANY_PATTERN = re.compile(
    r"\b(amazon|amzn|google|alphabet|googl|goog|apple|aapl|microsoft|msft|tesla|tsla|nvidia|nvda|meta|facebook|fb)\b",
    re.IGNORECASE,
)
DOC_TYPE_PATTERN = re.compile(
    r"\b(annual report|quarterly report|current report|10-k|10-q|8-k)\b",
    re.IGNORECASE,
)
TIME_PATTERN = re.compile(
    r"\b(20\d{2}|q[1-4]|quarter\s*[1-4]|fiscal\s+year|fy\s*20\d{2})\b",
    re.IGNORECASE,
)
FINANCIAL_PATTERN = re.compile(
    r"\b(revenue|net income|profit|profitability|cash flow|assets|liabilities|equity|eps|earnings)\b",
    re.IGNORECASE,
)

def _has_required_signals(user_query: str) -> bool:
    """
    Return True if query contains at least one expected financial/SEC signal
    (company, doc type, time hint, or core financial metric).
    """
    if not user_query or not user_query.strip():
        return False

    return any(
        pattern.search(user_query)
        for pattern in (COMPANY_PATTERN, DOC_TYPE_PATTERN, TIME_PATTERN, FINANCIAL_PATTERN)
    )

def extract_filters(user_query: str) -> dict:
    """
    Extract structured metadata filters from a user query using LLM.

    Expected filter keys: company_name, doc_type, fiscal_year, fiscal_quarter.
    Behavior:
    - If query has no relevant financial/SEC signals (regex pre-check), returns {}.
    - Otherwise invokes structured LLM extraction and returns non-null fields only.
    - On extraction failure, returns {}.
    """
    if not _has_required_signals(user_query):
        print("No relevant company/doc/time/financial signals found. Skipping extraction.")
        return {}

    llm_structured = LLM.with_structured_output(ChunkMetadata)
    USER_QUERY_PROMPT = f"""
Extract metadata filters from the query. Return None for fields not mentioned.

USER QUERY: {user_query}

COMPANY MAPPINGS:
- Amazon/AMZN -> amazon
- Google/Alphabet/GOOGL/GOOG -> google
- Apple/AAPL -> apple
- Microsoft/MSFT -> microsoft
- Tesla/TSLA -> tesla
- Nvidia/NVDA -> nvidia
- Meta/Facebook/FB -> meta

DOC TYPE:
- Annual report -> 10-k
- Quarterly report -> 10-q
- Current report -> 8-k

EXAMPLES:
"Amazon Q3 2024 revenue" -> {{"company_name": "amazon", "doc_type": "10-q", "fiscal_year": 2024, "fiscal_quarter": "q3"}}
"Apple 2023 annual report" -> {{"company_name": "apple", "doc_type": "10-k", "fiscal_year": 2023}}
"Tesla profitability" -> {{"company_name": "tesla"}}

Extract metadata:
""".strip()

    try:
        metadata = llm_structured.invoke(USER_QUERY_PROMPT)
        return metadata.model_dump(exclude_none=True)
    except Exception as e:
        print(f"extract_filters failed: {e}")
        return {}


user_query="Boom boom max berteeras4"
extract_filters(user_query=user_query)

No relevant company/doc/time/financial signals found. Skipping extraction.


{}

In [20]:
### GENERATE KEYWORDS

def generate_ranking_keywords(user_query: str):
    if not _has_required_signals(user_query):
        print("No relevant company/doc/time/financial signals found. Skipping extraction.")
        return {}

    user_query_prompt = f"""Generate EXACTLY 5 financial keywords from SEC filings terminology.

USER QUERY: {user_query}

USE EXACT TERMS FROM 10-K/10-Q FILINGS:

STATEMENT HEADINGS:
"consolidated statements of operations", "consolidated balance sheets", "consolidated statements of cash flows", "consolidated statements of stockholders equity"

INCOME STATEMENT:
"revenue", "net revenue", "cost of revenue", "gross profit", "operating income", "net income", "earnings per share"

BALANCE SHEET:
"total assets", "cash and cash equivalents", "total liabilities", "stockholders equity", "working capital", "long-term debt"

CASH FLOWS:
"cash flows from operating activities", "net cash provided by operating activities", "cash flows from investing activities", "free cash flow", "capital expenditures"

RULES:
- Return EXACTLY 5 keywords
- Use exact phrases from SEC filings
- Match query topic (revenue -> revenue terms, cash -> cash flow terms)
- Use "cash flows" (plural), "stockholders equity"

EXAMPLES:
"revenue analysis" -> ["revenue", "net revenue", "total revenue", "consolidated statements of operations", "net sales"]
"cash flow performance" -> ["consolidated statements of cash flows", "cash flows from operating activities", "net cash provided by operating activities", "free cash flow", "operating activities"]
"balance sheet strength" -> ["consolidated balance sheets", "total assets", "stockholders equity", "cash and cash equivalents", "long-term debt"]

Generate EXACTLY 5 keywords:
""".strip()

    try:
        llm_structured = LLM.with_structured_output(RankingKeywords)
        result = llm_structured.invoke(user_query_prompt)

        if hasattr(result, "keywords"):
            return result.keywords
        if hasattr(result, "ranked_keyword"):  # legacy schema fallback
            return [result.ranked_keyword]

        print(f"Unexpected schema: {result}")
        return []
    except Exception as e:
        print(f"generate_ranking_keywords failed: {type(e).__name__}: {e}")
        return []

generate_ranking_keywords("what is google's revenue in 2024?")

['revenue',
 'net income',
 'free cash flow',
 'cash flows from operating activities',
 'stockholders equity']

In [21]:
### SEARCH DOCS FROM VECTOR DB

def build_search_keyword_args(filters=None, ranking_keywords=None, k=5):
    """
    Build kwargs for Chroma search methods.

    Returns keys:
    - k, fetch_k
    - filter (metadata filter)
    - where_document (document text filter)
    """
    # Basic validation for top-k
    if not isinstance(k, int) or k <= 0:
        raise ValueError(f"`k` must be a positive integer. Got: {k}")

    search_keyword_args = {
        "k": k,
        "fetch_k": k * 20,  # wider candidate pool for better reranking/MMR
    }

    # ---- Metadata filters ----
    if filters is not None:
        if not isinstance(filters, dict):
            raise TypeError(f"`filters` must be a dict or None. Got: {type(filters).__name__}")

        # Remove empty values to avoid invalid/pointless filter clauses
        cleaned_filters = {
            key: value
            for key, value in filters.items()
            if value is not None and str(value).strip() != ""
        }

        if cleaned_filters:
            # Single filter -> direct filter
            if len(cleaned_filters) == 1:
                search_keyword_args["filter"] = cleaned_filters
            # Multiple filters -> AND all conditions
            else:
                search_keyword_args["filter"] = {
                    "$and": [{key: value} for key, value in cleaned_filters.items()]
                }

    # ---- Content keyword filters ----
    if ranking_keywords is not None:
        # check if the data-structure is correct or not
        if not isinstance(ranking_keywords, (list, tuple, set)):
            raise TypeError(
                f"`ranking_keywords` must be a list/tuple/set or None. Got: {type(ranking_keywords).__name__}"
            )

        # Normalize: keep non-empty strings only, preserve order, de-duplicate
        normalized_keywords = []
        seen = set()
        for kw in ranking_keywords:
            if not isinstance(kw, str):
                continue
            kw_clean = kw.strip()
            if kw_clean and kw_clean not in seen:
                normalized_keywords.append(kw_clean)
                seen.add(kw_clean)

        if normalized_keywords:
            if len(normalized_keywords) == 1:
                search_keyword_args["where_document"] = {"$contains": normalized_keywords[0]}
            else:
                search_keyword_args["where_document"] = {
                    "$or": [{"$contains": kw} for kw in normalized_keywords]
                }

    return search_keyword_args


In [22]:
from collections.abc import Sequence

### SEARCH DOCS


def search_vector_database(
    query: str,
    filters: dict | None = None,
    ranking_keywords: Sequence[str] | None = None,
    k: int = 3,
):
    """
    Search documents using metadata and content filters.

    Args:
        query: Search query text.
        filters: Optional metadata filters, e.g. {"company_name": "amazon", "fiscal_year": 2023}.
        ranking_keywords: Optional content keywords; at least one match is required when provided.
        k: Number of results to return.

    Returns:
        List of matching Document objects.
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("`query` must be a non-empty string.")

    search_kwargs = build_search_keyword_args(
        filters=filters,
        ranking_keywords=ranking_keywords,
        k=k,
    )

    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs=search_kwargs,
    )

    return retriever.invoke(query)


In [23]:
### EXTRACT DOCUMENTS HEADING AND SUB-HEADING FOR RE-RANKING

def extract_headings_with_content(text: str) -> list[str]:
    """
        Extract markdown headings with the immediately following paragraph.

        Args:
            text: Document text content.

        Returns:
            A list of extracted heading + content chunks.
    """
    try:
        if not isinstance(text, str):
            raise TypeError(f"`text` must be a string. Got: {type(text).__name__}")

        text = text.strip()
        if not text:
            return []

        extracted_chunks: list[str] = []
        sections = [section.strip() for section in text.split("\n\n") if section.strip()]
        heading_pattern = re.compile(r"^#+\s+")

        idx = 0
        while idx < len(sections):
            current_section = sections[idx]

            if heading_pattern.match(current_section):
                if idx + 1 < len(sections):
                    next_content = sections[idx + 1]
                    extracted_chunks.append(f"{current_section}\n\n{next_content}")
                    idx += 2
                else:
                    extracted_chunks.append(current_section)
                    idx += 1
            else:
                idx += 1

        return extracted_chunks

    except Exception as e:
        print(f"extract_headings_with_content failed: {type(e).__name__}: {e}")
        return []


In [24]:
def rank_documents_by_keywords(docs, keywords, k=5):
    """
    Rank documents using BM25Plus on heading + content chunks.

    Args:
        docs: List of Document objects.
        keywords: List of ranking keywords.
        k: Number of top documents to return.

    Returns:
        List of top-k Document objects sorted by BM25 score.
    """
    if not docs or not keywords:
        print("No documents or keywords found.")
        return []

    if not isinstance(k, int) or k <= 0:
        raise ValueError(f"`k` must be a positive integer. Got: {k}")

    def _tokenize(text: str) -> list[str]:
        return re.findall(r"\b\w+\b", text.lower())

    query_tokens = _tokenize(" ".join(str(kw) for kw in keywords if kw))

    if not query_tokens:
        print("No valid query tokens found.")
        return []

    doc_tokens = []
    for doc in docs:
        chunks = extract_headings_with_content(getattr(doc, "page_content", "") or "")
        combined_text = " ".join(chunks) if chunks else getattr(doc, "page_content", "") or ""
        doc_tokens.append(_tokenize(combined_text))

    if not any(doc_tokens):
        print("No valid document content found.")
        return []

    bm25 = BM25Plus(doc_tokens)
    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    for rank, idx in enumerate(ranked_indices[:k], 1):
        print(f"[{rank}] Doc {idx}: score={scores[idx]:.4f}")

    return [docs[i] for i in ranked_indices[:k]]

In [27]:
from types import SimpleNamespace


def test_all_pipeline_functions():
    """Run compact unit-style checks for all pipeline helper functions in this notebook."""

    # ----- _has_required_signals -----
    assert _has_required_signals("Amazon Q3 2024 revenue") is True
    assert _has_required_signals("hello world") is False
    assert _has_required_signals("   ") is False

    # ----- extract_filters (with LLM stub) -----
    class _FakeMeta:
        def model_dump(self, exclude_none=True):
            return {
                "company_name": "amazon",
                "doc_type": "10-q",
                "fiscal_year": 2024,
                "fiscal_quarter": "q3",
            }

    class _FakeStructuredLLM:
        def invoke(self, _prompt):
            return _FakeMeta()

    class _FakeLLM:
        def with_structured_output(self, _schema):
            return _FakeStructuredLLM()

    global LLM
    _old_llm = LLM
    LLM = _FakeLLM()
    try:
        f = extract_filters("Amazon Q3 2024 revenue")
        assert f["company_name"] == "amazon"
        assert f["doc_type"] == "10-q"
        assert f["fiscal_year"] == 2024
        assert extract_filters("random non-financial query") == {}
    finally:
        LLM = _old_llm

    # ----- generate_ranking_keywords (with LLM stub) -----
    class _KWResult:
        def __init__(self):
            self.keywords = [
                "revenue",
                "net revenue",
                "gross profit",
                "operating income",
                "net income",
            ]

    class _KWStructured:
        def invoke(self, _prompt):
            return _KWResult()

    class _KWLLM:
        def with_structured_output(self, _schema):
            return _KWStructured()

    _old_llm = LLM
    LLM = _KWLLM()
    try:
        kws = generate_ranking_keywords("what is google's revenue in 2024?")
        assert isinstance(kws, list) and len(kws) == 5
        assert "revenue" in kws
        assert generate_ranking_keywords("nonsense text") == {}
    finally:
        LLM = _old_llm

    # ----- build_search_keyword_args -----
    args = build_search_keyword_args(
        filters={"company_name": "google", "fiscal_year": 2024, "fiscal_quarter": ""},
        ranking_keywords=["revenue", "revenue", "net income", "  ", None],
        k=3,
    )
    assert args["k"] == 3 and args["fetch_k"] == 60
    assert "filter" in args and "$and" in args["filter"]
    assert "where_document" in args and "$or" in args["where_document"]

    # validation branches
    try:
        build_search_keyword_args(k=0)
        raise AssertionError("Expected ValueError for k=0")
    except ValueError:
        pass

    try:
        build_search_keyword_args(filters="bad")
        raise AssertionError("Expected TypeError for non-dict filters")
    except TypeError:
        pass

    try:
        build_search_keyword_args(ranking_keywords="bad")
        raise AssertionError("Expected TypeError for non-sequence ranking_keywords")
    except TypeError:
        pass

    # ----- search_vector_database (with vector store stub) -----
    class _FakeRetriever:
        def __init__(self, docs):
            self._docs = docs

        def invoke(self, query):
            assert isinstance(query, str)
            return self._docs

    class _FakeVectorStore:
        def as_retriever(self, search_type, search_kwargs):
            assert search_type == "mmr"
            assert search_kwargs["k"] == 2
            return _FakeRetriever([SimpleNamespace(page_content="doc one")])

    global vector_store
    _old_vs = vector_store
    vector_store = _FakeVectorStore()
    try:
        out = search_vector_database("revenue query", filters={"company_name": "google"}, k=2)
        assert len(out) == 1

        try:
            search_vector_database("   ")
            raise AssertionError("Expected ValueError for empty query")
        except ValueError:
            pass
    finally:
        vector_store = _old_vs

    # ----- extract_headings_with_content -----
    md = "# Revenue\n\nRevenue grew 10%.\n\n## Net Income\n\nNet income increased."
    chunks = extract_headings_with_content(md)
    assert isinstance(chunks, list) and len(chunks) == 2
    assert "# Revenue" in chunks[0]

    assert extract_headings_with_content("") == []
    assert extract_headings_with_content(None) == []

    # ----- rank_documents_by_keywords -----
    docs = [
        SimpleNamespace(page_content="# Revenue\n\nRevenue and net income improved."),
        SimpleNamespace(page_content="# Cash Flows\n\nCash flows from operating activities declined."),
        SimpleNamespace(page_content="# Risk Factors\n\nMarket volatility remained high."),
    ]

    ranked = rank_documents_by_keywords(docs, ["revenue", "net income"], k=2)
    assert isinstance(ranked, list) and len(ranked) == 2
    assert rank_documents_by_keywords([], ["revenue"], k=2) == []
    assert rank_documents_by_keywords(docs, [], k=2) == []

    try:
        rank_documents_by_keywords(docs, ["revenue"], k=0)
        raise AssertionError("Expected ValueError for invalid k")
    except ValueError:
        pass

    print("All tests passed.")

In [28]:
# Run all notebook function tests
test_all_pipeline_functions()

No relevant company/doc/time/financial signals found. Skipping extraction.
No relevant company/doc/time/financial signals found. Skipping extraction.
extract_headings_with_content failed: TypeError: `text` must be a string. Got: NoneType
[1] Doc 0: score=9.1083
[2] Doc 1: score=4.1589
No documents or keywords found.
No documents or keywords found.
All tests passed.


In [ ]:
from docling.document_converter import DocumentConverter

def pdf_to_markdown(pdf_path: str, output_md_path: str):
    """
    Convert a PDF file to Markdown using Docling.
    """
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    markdown_content = result.document.export_to_markdown()

    with open(output_md_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    print(f"Markdown file saved at: {output_md_path}")

input_path = "/home/siddhant-gond/Desktop/kpi/docs/UI_docs/CortexAD-Design & Theme Guideline 1.pdf"
output_md = "/home/siddhant-gond/Desktop/kpi/docs/UI_docs/CortexAD-Design & Theme Guideline 1.md"

pdf_to_markdown(input_path, output_md)
